In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
branch_table = dbutils.widgets.get("branch_table")
office_table = dbutils.widgets.get("office_table")
clientepisodefsall_table = dbutils.widgets.get("clientepisodefsall_table")
clientepisodediagprocs_table = dbutils.widgets.get("clientepisodediagprocs_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW diagnosis_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(Sequence AS INT) AS Sequence,
  CAST(DiagCode AS STRING) AS DiagCode,
  NULL AS DiagDesc,
  NULL AS Version,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey
FROM (
  WITH 
  diagnosis_cte AS (
    SELECT
    CAST('{fetch_date}' AS DATE) AS ReportingDate,
    CASE 
        WHEN b.branch_code RLIKE '[A-Za-z]' THEN ofc.OfficeNumber 
        ELSE b.branch_code 
    END AS FacilityCode,
    bi_h.i_id AS AcctNbr,
    1 AS Sequence,
    -- ROW_NUMBER() OVER (PARTITION BY bi_h.i_id ORDER BY cd.ced_SortOrder) AS Sequence,
    cd.ced_diICDCode AS DiagCode,
    '6' AS SourceSystemKey
    FROM {source_table} bi_h
    JOIN {branch_table} b 
        ON bi_h.i_branchcode = b.branch_code
    LEFT JOIN {office_table} ofc
        ON ofc.OfficeAbbreviation = b.branch_code
    JOIN {clientepisodefsall_table} cefs
        ON cefs.cefs_id = bi_h.i_cefsid
    JOIN {clientepisodediagprocs_table} cd
        ON cd.ced_epiid = cefs.cefs_epiid
    WHERE bi_h.i_Balance <> 0
  ),
  diagnosis_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM diagnosis_cte
  )
  SELECT 
    ReportingDate,
    FacilityCode,
    AcctNbr,
    Sequence,
    DiagCode,
    SourceSystemKey
  FROM diagnosis_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING diagnosis_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 6

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.Sequence = src.Sequence,
    tgt.DiagCode = src.DiagCode,
    tgt.DiagDesc = src.DiagDesc,
    tgt.Version = src.Version,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    Sequence,
    DiagCode,
    DiagDesc,
    Version,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.Sequence,
    src.DiagCode,
    src.DiagDesc,
    src.Version,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)